In [3]:
import numpy as np
import pandas as pd
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import warnings
warnings.filterwarnings("ignore")

# ==================== AF MODEL DEFINITION (ساده شده برای داده جدولی) ====================

class AFFeatureExtractor(nn.Module):
    """
    مدل AF با معماری MLP (مناسب برای داده جدولی)
    اما حفظ ساختار سه بخشی Feature Extractor + Classifier + Discriminator
    و روش آموزش مشابه کد اصلی (سه نرخ یادگیری جداگانه + ذخیره/بازیابی وزن‌ها)
    """
    def __init__(self, input_dim, num_classes, embedding_size=512, hidden_dims=[256, 128]):
        super().__init__()
        
        self.input_dim = input_dim
        self.embedding_size = embedding_size
        
        # ========== Feature Extractor (MLP با BatchNorm و Dropout) ==========
        feature_layers = []
        prev_dim = input_dim
        
        for i, h_dim in enumerate(hidden_dims):
            feature_layers.extend([
                nn.Linear(prev_dim, h_dim),
                nn.BatchNorm1d(h_dim),
                nn.ELU(),  # مثل کد اصلی
                nn.Dropout(0.1)
            ])
            prev_dim = h_dim
        
        # Embedding layer
        feature_layers.extend([
            nn.Linear(prev_dim, embedding_size),
            nn.BatchNorm1d(embedding_size)
        ])
        
        self.feature_extractor = nn.Sequential(*feature_layers)
        
        # ========== Classifier (روی بردار ویژگی) ==========
        self.classifier = nn.Sequential(
            nn.Linear(embedding_size, 512),
            nn.BatchNorm1d(512),
            nn.ELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes),
            nn.BatchNorm1d(num_classes)
        )
        
        # ========== Discriminator (روی بردار ویژگی) ==========
        self.discriminator = nn.Sequential(
            nn.Linear(embedding_size, 512),
            nn.BatchNorm1d(512),
            nn.ELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
        
        # جدا کردن پارامترها برای بهینه‌سازهای مختلف (مثل کد اصلی)
        self.classifier_params = list(self.classifier.parameters())
        self.discriminator_params = list(self.discriminator.parameters())
        self.feature_params = list(self.feature_extractor.parameters())
        
        # نرخ‌های یادگیری متفاوت (مانند کد اصلی)
        self.lr_classifier = 1e-4      # نرخ بالاتر برای طبقه‌بند
        self.lr_discriminator = 1e-5   # نرخ پایین‌تر برای تمایزگر
        self.lr_combined = 1e-5        # نرخ متوسط برای مدل ترکیبی
        
        self.class_loss_weight = 4.0
        self.dis_loss_weight = 4.0
    
    def forward(self, x, return_features=False):
        """forward معمولی برای inference"""
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        class_output = nn.functional.softmax(class_output, dim=1)
        
        if return_features:
            return class_output, features
        return class_output
    
    def forward_with_domain(self, x):
        """forward با خروجی طبقه‌بند و تمایزگر (برای آموزش)"""
        features = self.feature_extractor(x)
        class_output = self.classifier(features)
        domain_output = self.discriminator(features)
        return class_output, domain_output
    
    def predict(self, x, device='cpu'):
        """پیش‌بینی برای یک یا چند نمونه"""
        self.eval()
        with torch.no_grad():
            if len(x.shape) == 1:
                x_tensor = torch.FloatTensor(x).unsqueeze(0).to(device)
            else:
                x_tensor = torch.FloatTensor(x).to(device)
            class_output = self.forward(x_tensor)
            pred = torch.argmax(class_output, dim=1).cpu().numpy()
            if len(pred) == 1:
                return pred[0]
            return pred
    
    def partial_fit_batch(self, X_source, y_source, X_target, device='cpu', 
                          class_loss_weight=4.0, dis_loss_weight=4.0):
        """
        آموزش یک بچ به روش کد اصلی (ADA):
        1. ذخیره وزن‌های discriminator
        2. آموزش combined model (classifier + feature extractor) برای فریب discriminator
        3. برگرداندن وزن‌های discriminator
        4. آموزش جداگانه discriminator با وزن‌های ثابت برای feature extractor
        """
        self.train()
        
        # تبدیل به tensor
        X_s = torch.FloatTensor(X_source).to(device)
        y_s = torch.LongTensor(y_source).to(device)
        X_t = torch.FloatTensor(X_target).to(device)
        
        batch_size = min(X_s.size(0), X_t.size(0))
        X_s = X_s[:batch_size]
        y_s = y_s[:batch_size]
        X_t = X_t[:batch_size]
        
        X_combined = torch.cat([X_s, X_t], dim=0)
        
        # ====== مرحله 1: ذخیره وزن‌های discriminator ======
        disc_weights = copy.deepcopy(self.discriminator.state_dict())
        
        # ====== مرحله 2: آموزش combined model ======
        # بهینه‌ساز برای feature extractor + classifier
        opt_combined = optim.AdamW(self.feature_params + self.classifier_params, 
                                   lr=self.lr_combined, betas=(0.9, 0.999), weight_decay=1e-4)
        opt_combined.zero_grad()
        
        class_output, domain_output = self.forward_with_domain(X_combined)
        
        # Classifier loss (فقط روی داده منبع)
        class_loss = nn.functional.cross_entropy(class_output[:batch_size], y_s)
        
        # Domain loss برای combined (هدف: فریب discriminator)
        # برچسب‌ها معکوس: می‌خواهیم discriminator منبع را هدف و هدف را منبع تشخیص دهد
        domain_labels_combined = torch.cat([
            torch.ones(batch_size, dtype=torch.float32).to(device),   # منبع -> برچسب 1 (هدف)
            torch.zeros(batch_size, dtype=torch.float32).to(device)   # هدف -> برچسب 0 (منبع)
        ]).unsqueeze(1)
        domain_loss_combined = nn.functional.binary_cross_entropy(domain_output, domain_labels_combined)
        
        # کل loss (weighted)
        total_loss_combined = class_loss_weight * class_loss - dis_loss_weight * domain_loss_combined
        total_loss_combined.backward()
        torch.nn.utils.clip_grad_norm_(self.parameters(), max_norm=0.5)
        opt_combined.step()
        
        # ====== مرحله 3: برگرداندن وزن‌های discriminator ======
        self.discriminator.load_state_dict(disc_weights)
        
        # ====== مرحله 4: آموزش جداگانه discriminator ======
        opt_discriminator = optim.AdamW(self.discriminator_params, lr=self.lr_discriminator,
                                        betas=(0.9, 0.999), weight_decay=1e-4)
        opt_discriminator.zero_grad()
        
        # استخراج ویژگی با مدل فعلی (بدون به‌روزرسانی گرادیان)
        with torch.no_grad():
            features_s = self.feature_extractor(X_s)
            features_t = self.feature_extractor(X_t)
        
        features_combined = torch.cat([features_s, features_t], dim=0)
        
        # Discriminator روی ویژگی‌های ثابت
        domain_output_new = self.discriminator(features_combined)
        
        # برچسب‌های واقعی برای discriminator: منبع=0، هدف=1
        domain_labels_disc = torch.cat([
            torch.zeros(batch_size, dtype=torch.float32).to(device),
            torch.ones(batch_size, dtype=torch.float32).to(device)
        ]).unsqueeze(1)
        domain_loss_disc = nn.functional.binary_cross_entropy(domain_output_new, domain_labels_disc)
        domain_loss_disc.backward()
        torch.nn.utils.clip_grad_norm_(self.discriminator_params, max_norm=0.5)
        opt_discriminator.step()
        
        return {
            'class_loss': class_loss.item(),
            'domain_loss_combined': domain_loss_combined.item(),
            'domain_loss_disc': domain_loss_disc.item()
        }


class AFClassifier:
    """
    Wrapper کلاس AF با interface مشابه کد دوم
    """
    def __init__(self, input_dim=None, num_classes=None, embedding_size=512, 
                 hidden_dims=[256, 128], seed=42, 
                 class_loss_weight=4.0, dis_loss_weight=4.0):
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.embedding_size = embedding_size
        self.hidden_dims = hidden_dims
        self.seed = seed
        self.class_loss_weight = class_loss_weight
        self.dis_loss_weight = dis_loss_weight
        self.model = None
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.is_fitted = False
        
        # بافر برای آموزش بچ (برای online learning)
        self.buffer_X = []
        self.buffer_y = []
        self.buffer_X_target = []
    
    def _initialize_model(self):
        """ایجاد مدل"""
        if self.input_dim is None or self.num_classes is None:
            raise ValueError("input_dim and num_classes must be set")
        
        torch.manual_seed(self.seed)
        self.model = AFFeatureExtractor(
            input_dim=self.input_dim,
            num_classes=self.num_classes,
            embedding_size=self.embedding_size,
            hidden_dims=self.hidden_dims
        ).to(self.device)
        self.is_fitted = True
        
        print(f"  Model initialized: input_dim={self.input_dim}, num_classes={self.num_classes}")
    
    def learn_one(self, x, y):
        """یادگیری از یک نمونه (online learning با بافر)"""
        if not self.is_fitted:
            self.input_dim = len(x)
            self.num_classes = max(y + 1, 2)
            self._initialize_model()
        
        # تبدیل ورودی
        if isinstance(x, dict):
            x = np.array([x[i] for i in range(len(x))])
        
        # اضافه کردن به بافر
        self.buffer_X.append(x)
        self.buffer_y.append(y)
        self.buffer_X_target.append(x)
        
        # آموزش بچ زمانی که بافر پر شد
        if len(self.buffer_X) >= 32:
            X_batch = np.array(self.buffer_X)
            y_batch = np.array(self.buffer_y)
            X_target_batch = np.array(self.buffer_X_target)
            
            _ = self.model.partial_fit_batch(
                X_batch, y_batch, X_target_batch, 
                device=self.device,
                class_loss_weight=self.class_loss_weight,
                dis_loss_weight=self.dis_loss_weight
            )
            
            # خالی کردن بافر
            self.buffer_X = []
            self.buffer_y = []
            self.buffer_X_target = []
        
        return self
    
    def predict_one(self, x):
        """پیش‌بینی برای یک نمونه"""
        if not self.is_fitted:
            return 0
        
        if isinstance(x, dict):
            x = np.array([x[i] for i in range(len(x))])
        
        return self.model.predict(x, device=self.device)
    
    def train_batch_mode(self, X_source, y_source, X_target, n_epochs=80, batch_size=32):
        """آموزش بچ (برای مدل پایه) به روش کد اصلی"""
        if not self.is_fitted:
            self.input_dim = X_source.shape[1]
            self.num_classes = len(np.unique(y_source))
            self._initialize_model()
        
        n_samples = len(X_source)
        
        print(f"  Training: {n_samples} samples, batch_size={batch_size}, epochs={n_epochs}")
        
        for epoch in range(n_epochs):
            indices = np.random.permutation(n_samples)
            total_class_loss = 0
            total_domain_loss_combined = 0
            total_domain_loss_disc = 0
            n_batches = 0
            
            for i in range(0, n_samples, batch_size):
                batch_indices = indices[i:i+batch_size]
                X_s_batch = X_source[batch_indices]
                y_s_batch = y_source[batch_indices]
                
                # انتخاب target sample (از داده منبع به عنوان target)
                target_indices = np.random.choice(n_samples, len(batch_indices), replace=False)
                X_t_batch = X_source[target_indices]
                
                losses = self.model.partial_fit_batch(
                    X_s_batch, y_s_batch, X_t_batch,
                    device=self.device,
                    class_loss_weight=self.class_loss_weight,
                    dis_loss_weight=self.dis_loss_weight
                )
                
                total_class_loss += losses['class_loss']
                total_domain_loss_combined += losses['domain_loss_combined']
                total_domain_loss_disc += losses['domain_loss_disc']
                n_batches += 1
            
            if (epoch + 1) % 20 == 0:
                print(f"  Epoch {epoch+1:3d}: Class Loss={total_class_loss/n_batches:.4f}, "
                      f"Domain Comb={total_domain_loss_combined/n_batches:.4f}, "
                      f"Domain Disc={total_domain_loss_disc/n_batches:.4f}")
        
        return self


# ==================== DATA LOADING FUNCTIONS ====================

def load_data(dataset_path='Dataset', csv_filename='ISCX_TOR_original.csv'):
    """بارگذاری داده"""
    original_df = pd.read_csv(f'{dataset_path}/{csv_filename}')
    train_df, temp_df = train_test_split(original_df, test_size=0.4, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    return train_df, val_df, test_df


def train_base_model(train_df, val_df):
    """آموزش مدل پایه AF"""
    X_train = train_df.iloc[:, :-1].values
    y_train = train_df.iloc[:, -1].values
    X_val = val_df.iloc[:, :-1].values
    y_val = val_df.iloc[:, -1].values
    
    # Encoding
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_val_encoded = label_encoder.transform(y_val)
    
    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    input_dim = X_train_scaled.shape[1]
    num_classes = len(label_encoder.classes_)
    
    print(f"\n{'='*60}")
    print(f"INITIALIZING AF MODEL (ADA - Adversarial Domain Adaptation)")
    print(f"{'='*60}")
    print(f"  Input dimension: {input_dim}")
    print(f"  Number of classes: {num_classes}")
    print(f"  Embedding size: 512")
    print(f"  Hidden dims: [256, 128]")
    print(f"  Class loss weight: {4.0}, Domain loss weight: {4.0}")
    print(f"  Learning rates: Classifier=1e-4, Discriminator=1e-5, Combined=1e-5")
    
    # ایجاد مدل
    model = AFClassifier(
        input_dim=input_dim,
        num_classes=num_classes,
        embedding_size=512,
        hidden_dims=[256, 128],
        class_loss_weight=4.0,
        dis_loss_weight=4.0,
        seed=42
    )
    
    # آموزش
    print("\nTraining AF model (batch mode with domain adaptation)...")
    model.train_batch_mode(X_train_scaled, y_train_encoded, X_train_scaled, n_epochs=80, batch_size=32)
    
    # ارزیابی روی validation
    model.model.eval()
    with torch.no_grad():
        X_val_tensor = torch.FloatTensor(X_val_scaled).to(model.device)
        val_output = model.model.forward(X_val_tensor)
        y_val_pred = torch.argmax(val_output, dim=1).cpu().numpy()
    
    val_accuracy = accuracy_score(y_val_encoded, y_val_pred)
    
    print(f"\n{'='*60}")
    print(f"BASE MODEL TRAINING COMPLETE")
    print(f"{'='*60}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    print(f"Number of classes: {num_classes}")
    
    return model, scaler, label_encoder


def load_perturbation_data(base_path='Dataset', attack_type='FGSM', test_perturb_levels=[0.1, 0.5, 1.0, 2.0, 5.0], 
                          scaler=None, label_encoder=None):
    """بارگذاری داده‌های perturbed"""
    datasets = []
    
    for perturb_level in sorted(test_perturb_levels):
        file_path = f'{base_path}/{attack_type}/{attack_type}_eps_{perturb_level}.csv'
        if not os.path.exists(file_path):
            print(f"Warning: {file_path} not found, skipping...")
            continue
            
        df = pd.read_csv(file_path)
        X = df.iloc[:, :-1].values
        y = df.iloc[:, -1].values
        
        y_encoded = label_encoder.transform(y)
        X_scaled = scaler.transform(X)
        
        datasets.append({
            'X': X_scaled,
            'y': y_encoded,
            'perturb_level': perturb_level
        })
        print(f"  Loaded {attack_type} eps={perturb_level}: {len(X_scaled)} samples")
    
    return datasets


def test_online_with_budget(base_model, X_test, y_test, perturb_level, update_budget=1.0):
    """تست آنلاین با بودجه (مثل کد دوم) - با خروجی 4 سنجه"""
    model = copy.deepcopy(base_model)
    
    n_samples = len(X_test)
    n_updates_allowed = int(n_samples * update_budget)
    
    y_pred = []
    y_true_list = []
    updates_done = 0
    
    print(f"\n--- Testing Perturbation Level {perturb_level} (Budget: {update_budget*100}%) ---")
    
    for i, (xi, yi) in enumerate(zip(X_test, y_test)):
        xi_dict = {j: float(x) for j, x in enumerate(xi)}
        
        pred = model.predict_one(xi_dict)
        y_pred.append(pred)
        y_true_list.append(yi)
        
        if updates_done < n_updates_allowed:
            model.learn_one(xi_dict, yi)
            updates_done += 1
        
        if (i + 1) % 1000 == 0 and i > 0:
            current_acc = accuracy_score(y_true_list[-1000:], y_pred[-1000:])
            print(f"  Sample {i+1}: Updates={updates_done}, Recent Acc={current_acc:.4f}")
    
    # محاسبه 4 سنجه اصلی
    accuracy = accuracy_score(y_true_list, y_pred)
    precision = precision_score(y_true_list, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true_list, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true_list, y_pred, average='weighted', zero_division=0)
    
    print(f"  Final - Acc: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    
    return {
        'Perturbation Level': perturb_level,
        'Update Budget %': update_budget * 100,
        'Total Samples': n_samples,
        'Updates Done': updates_done,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1 Score': f1
    }


def main_single_budget(attack_type='FGSM', test_perturb_levels=[0.1, 0.5, 1.0, 2.0, 5.0], update_budget=0.8):
    """اجرای اصلی با یک بودجه - نمایش 4 سنجه"""
    
    print("\n" + "="*80)
    print("AF MODEL (ADA - Adversarial Domain Adaptation) - ONLINE TESTING")
    print(f"Attack Type: {attack_type}, Update Budget: {update_budget*100}%")
    print("="*80)
    
    # بارگذاری داده
    print("\n[1] Loading data...")
    train_df, val_df, test_df = load_data()
    print(f"  Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
    
    # آموزش مدل پایه
    print("\n[2] Training base model...")
    base_model, scaler, label_encoder = train_base_model(train_df, val_df)
    
    # بارگذاری داده‌های perturbed
    print(f"\n[3] Loading perturbed data (attack type: {attack_type})...")
    perturb_datasets = load_perturbation_data(
        base_path='Dataset',
        attack_type=attack_type,
        test_perturb_levels=test_perturb_levels,
        scaler=scaler,
        label_encoder=label_encoder
    )
    
    if len(perturb_datasets) == 0:
        print(f"No perturbed datasets found for attack type: {attack_type}")
        return
    
    # تست
    print(f"\n[4] Running online tests with {update_budget*100}% update budget...")
    results = []
    for dataset in perturb_datasets:
        result = test_online_with_budget(
            base_model,
            dataset['X'],
            dataset['y'],
            dataset['perturb_level'],
            update_budget
        )
        results.append(result)
    
    # ذخیره نتایج با 4 سنجه
    df_results = pd.DataFrame(results)
    print("\n" + "="*60)
    print("FINAL RESULTS SUMMARY (4 Metrics)")
    print("="*60)
    print(df_results[['Perturbation Level', 'Updates Done', 'Accuracy', 'Precision', 'Recall', 'F1 Score']].to_string(index=False))
    
    if not os.path.exists('Results'):
        os.makedirs('Results')
    
    output_file = f'Results/af_ada_{attack_type}_budget_{int(update_budget*100)}.csv'
    df_results.to_csv(output_file, index=False)
    print(f"\nResults saved to: {output_file}")
    
    return df_results


def test_multiple_budgets(attack_type='FGSM', perturb_levels=[0.1, 0.5, 1.0, 2.0, 5.0]):
    """تست با چند بودجه مختلف - نمایش 4 سنجه"""
    budgets = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    all_results = []
    
    print("\n" + "="*80)
    print("AF MODEL - MULTIPLE BUDGETS TEST (4 Metrics)")
    print("="*80)
    
    # بارگذاری داده و آموزش مدل پایه (یک بار)
    train_df, val_df, test_df = load_data()
    base_model, scaler, label_encoder = train_base_model(train_df, val_df)
    
    perturb_datasets = load_perturbation_data(
        attack_type=attack_type,
        test_perturb_levels=perturb_levels,
        scaler=scaler,
        label_encoder=label_encoder
    )
    
    if len(perturb_datasets) == 0:
        print(f"No perturbed datasets found for attack type: {attack_type}")
        return
    
    for budget in budgets:
        print(f"\n{'='*60}")
        print(f"BUDGET: {budget*100}%")
        print('='*60)
        
        for dataset in perturb_datasets:
            result = test_online_with_budget(
                base_model,
                dataset['X'],
                dataset['y'],
                dataset['perturb_level'],
                budget
            )
            all_results.append(result)
            print(f"  Level {result['Perturbation Level']}: Acc={result['Accuracy']:.4f}, "
                  f"Prec={result['Precision']:.4f}, Rec={result['Recall']:.4f}, F1={result['F1 Score']:.4f}")
    
    # ذخیره و خلاصه
    df_all = pd.DataFrame(all_results)
    os.makedirs('Results', exist_ok=True)
    df_all.to_csv(f'Results/af_ada_{attack_type}_all_budgets.csv', index=False)
    
    # خلاصه Accuracy
    summary_acc = df_all.pivot_table(
        index='Perturbation Level',
        columns='Update Budget %',
        values='Accuracy',
        aggfunc='first'
    )
    
    # خلاصه F1
    summary_f1 = df_all.pivot_table(
        index='Perturbation Level',
        columns='Update Budget %',
        values='F1 Score',
        aggfunc='first'
    )
    
    print("\n" + "="*60)
    print("ACCURACY SUMMARY TABLE")
    print("="*60)
    print(summary_acc.round(4))
    
    print("\n" + "="*60)
    print("F1 SCORE SUMMARY TABLE")
    print("="*60)
    print(summary_f1.round(4))
    
    summary_acc.to_csv(f'Results/af_ada_{attack_type}_accuracy_summary.csv')
    summary_f1.to_csv(f'Results/af_ada_{attack_type}_f1_summary.csv')
    
    return df_all


def main(ATTACK_TYPE, TEST_PERTURB_LEVELS, UPDATE_BUDGET):
    """اجرای اصلی با نمایش 4 سنجه Acc, Precision, Recall, F1"""
    
   
    # ========== اجرا ==========
    print("="*80)
    print("ADA (Adversarial Domain Adaptation) - Original Paper Method")
    print("Model: Feature Extractor (MLP) + Classifier + Domain Discriminator")
    print("Activation: ELU")
    print("Training: Separate optimizers (3 different learning rates)")
    print("Strategy: Weight saving/restoring like original implementation")
    print("")
    print("EVALUATION METRICS: Accuracy, Precision, Recall, F1-Score")
    print("="*80)
    
    # تست با یک بودجه
    results_single = main_single_budget(ATTACK_TYPE, TEST_PERTURB_LEVELS, UPDATE_BUDGET)
    
    # (اختیاری) تست با چند بودجه - خط زیر را uncomment کنید
    # results_multiple = test_multiple_budgets(ATTACK_TYPE, TEST_PERTURB_LEVELS)




In [4]:
if __name__ == "__main__":
        # ========== تنظیمات ==========
    ATTACK_TYPE = 'FGSM'  # گزینه‌ها: 'DeepFool', 'PGD', 'CW'
    TEST_PERTURB_LEVELS = [0.01, 0.05, 0.1, 0.2, 0.5]
    UPDATE_BUDGET = 0.2  # 20% بودجه برای به‌روزرسانی
    main(ATTACK_TYPE, TEST_PERTURB_LEVELS, UPDATE_BUDGET)

ADA (Adversarial Domain Adaptation) - Original Paper Method
Model: Feature Extractor (MLP) + Classifier + Domain Discriminator
Activation: ELU
Training: Separate optimizers (3 different learning rates)
Strategy: Weight saving/restoring like original implementation

EVALUATION METRICS: Accuracy, Precision, Recall, F1-Score

AF MODEL (ADA - Adversarial Domain Adaptation) - ONLINE TESTING
Attack Type: FGSM, Update Budget: 20.0%

[1] Loading data...
  Train: 8648, Val: 2883, Test: 2883

[2] Training base model...

INITIALIZING AF MODEL (ADA - Adversarial Domain Adaptation)
  Input dimension: 36
  Number of classes: 8
  Embedding size: 512
  Hidden dims: [256, 128]
  Class loss weight: 4.0, Domain loss weight: 4.0
  Learning rates: Classifier=1e-4, Discriminator=1e-5, Combined=1e-5

Training AF model (batch mode with domain adaptation)...
  Model initialized: input_dim=36, num_classes=8
  Training: 8648 samples, batch_size=32, epochs=80
  Epoch  20: Class Loss=0.8313, Domain Comb=0.7111, Do

In [5]:
if __name__ == "__main__":
        # ========== تنظیمات ==========
    ATTACK_TYPE = 'PGD'  # گزینه‌ها: 'DeepFool', 'PGD', 'CW'
    TEST_PERTURB_LEVELS = [0.01, 0.05, 0.1, 0.2, 0.5]
    UPDATE_BUDGET = 0.2  # 20% بودجه برای به‌روزرسانی
    main(ATTACK_TYPE, TEST_PERTURB_LEVELS, UPDATE_BUDGET)

ADA (Adversarial Domain Adaptation) - Original Paper Method
Model: Feature Extractor (MLP) + Classifier + Domain Discriminator
Activation: ELU
Training: Separate optimizers (3 different learning rates)
Strategy: Weight saving/restoring like original implementation

EVALUATION METRICS: Accuracy, Precision, Recall, F1-Score

AF MODEL (ADA - Adversarial Domain Adaptation) - ONLINE TESTING
Attack Type: PGD, Update Budget: 20.0%

[1] Loading data...
  Train: 8648, Val: 2883, Test: 2883

[2] Training base model...

INITIALIZING AF MODEL (ADA - Adversarial Domain Adaptation)
  Input dimension: 36
  Number of classes: 8
  Embedding size: 512
  Hidden dims: [256, 128]
  Class loss weight: 4.0, Domain loss weight: 4.0
  Learning rates: Classifier=1e-4, Discriminator=1e-5, Combined=1e-5

Training AF model (batch mode with domain adaptation)...
  Model initialized: input_dim=36, num_classes=8
  Training: 8648 samples, batch_size=32, epochs=80
  Epoch  20: Class Loss=0.8385, Domain Comb=0.7100, Dom

In [8]:
if __name__ == "__main__":
        # ========== تنظیمات ==========
    ATTACK_TYPE = 'CW'  # گزینه‌ها: 'DeepFool', 'PGD', 'CW'
    TEST_PERTURB_LEVELS = [0.1, 0.5, 1.0, 2.0, 5.0]
    UPDATE_BUDGET = 0.2  # 20% بودجه برای به‌روزرسانی
    main(ATTACK_TYPE, TEST_PERTURB_LEVELS, UPDATE_BUDGET)

ADA (Adversarial Domain Adaptation) - Original Paper Method
Model: Feature Extractor (MLP) + Classifier + Domain Discriminator
Activation: ELU
Training: Separate optimizers (3 different learning rates)
Strategy: Weight saving/restoring like original implementation

EVALUATION METRICS: Accuracy, Precision, Recall, F1-Score

AF MODEL (ADA - Adversarial Domain Adaptation) - ONLINE TESTING
Attack Type: CW, Update Budget: 20.0%

[1] Loading data...
  Train: 8648, Val: 2883, Test: 2883

[2] Training base model...

INITIALIZING AF MODEL (ADA - Adversarial Domain Adaptation)
  Input dimension: 36
  Number of classes: 8
  Embedding size: 512
  Hidden dims: [256, 128]
  Class loss weight: 4.0, Domain loss weight: 4.0
  Learning rates: Classifier=1e-4, Discriminator=1e-5, Combined=1e-5

Training AF model (batch mode with domain adaptation)...
  Model initialized: input_dim=36, num_classes=8
  Training: 8648 samples, batch_size=32, epochs=80
  Epoch  20: Class Loss=0.8284, Domain Comb=0.7104, Doma

In [9]:
if __name__ == "__main__":
        # ========== تنظیمات ==========
    ATTACK_TYPE = 'DeepFool'  # گزینه‌ها: 'DeepFool', 'PGD', 'CW'
    TEST_PERTURB_LEVELS = [0.1, 0.5, 1.0, 2.0, 5.0]
    UPDATE_BUDGET = 0.2  # 20% بودجه برای به‌روزرسانی
    main(ATTACK_TYPE, TEST_PERTURB_LEVELS, UPDATE_BUDGET)

ADA (Adversarial Domain Adaptation) - Original Paper Method
Model: Feature Extractor (MLP) + Classifier + Domain Discriminator
Activation: ELU
Training: Separate optimizers (3 different learning rates)
Strategy: Weight saving/restoring like original implementation

EVALUATION METRICS: Accuracy, Precision, Recall, F1-Score

AF MODEL (ADA - Adversarial Domain Adaptation) - ONLINE TESTING
Attack Type: DeepFool, Update Budget: 20.0%

[1] Loading data...
  Train: 8648, Val: 2883, Test: 2883

[2] Training base model...

INITIALIZING AF MODEL (ADA - Adversarial Domain Adaptation)
  Input dimension: 36
  Number of classes: 8
  Embedding size: 512
  Hidden dims: [256, 128]
  Class loss weight: 4.0, Domain loss weight: 4.0
  Learning rates: Classifier=1e-4, Discriminator=1e-5, Combined=1e-5

Training AF model (batch mode with domain adaptation)...
  Model initialized: input_dim=36, num_classes=8
  Training: 8648 samples, batch_size=32, epochs=80
  Epoch  20: Class Loss=0.8297, Domain Comb=0.7107